# SNE Benchmark Analysis

In [11]:
import re
from pathlib import Path
import pandas as pd

BENCHMARK_DIR = Path("../../../logs/remote/sne_benchmark")
OUTPUT_CSV = Path("aggregated_results.csv")

result_csvs = sorted(BENCHMARK_DIR.glob("*/logs/runs_benchmark/*/job_*_results.csv"))
print(f"Found {len(result_csvs)} result files:")
for p in result_csvs:
    print(f"  {p}")

frames = []
for csv_path in result_csvs:
    job_match = re.search(r"job_(\d+)_results", csv_path.name)
    job_id = int(job_match.group(1)) if job_match else None
    df = pd.read_csv(csv_path)
    df.insert(0, "job_id", job_id)
    df.insert(1, "source_dir", csv_path.parts[-5])  # sne_benchmark_XXXXXXX_N folder
    frames.append(df)

aggregated = pd.concat(frames, ignore_index=True).sort_values("job_id").reset_index(drop=True)

aggregated.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved {len(aggregated)} rows to {OUTPUT_CSV.resolve()}")
aggregated

Found 24 result files:
  ../../../logs/remote/sne_benchmark/sne_benchmark_2817861_7/logs/runs_benchmark/sne_grid_search/job_7_results.csv
  ../../../logs/remote/sne_benchmark/sne_benchmark_2817862_0/logs/runs_benchmark/sne_grid_search/job_0_results.csv
  ../../../logs/remote/sne_benchmark/sne_benchmark_2817863_1/logs/runs_benchmark/sne_grid_search/job_1_results.csv
  ../../../logs/remote/sne_benchmark/sne_benchmark_2817864_2/logs/runs_benchmark/sne_grid_search/job_2_results.csv
  ../../../logs/remote/sne_benchmark/sne_benchmark_2817865_3/logs/runs_benchmark/sne_grid_search/job_3_results.csv
  ../../../logs/remote/sne_benchmark/sne_benchmark_2817870_4/logs/runs_benchmark/sne_grid_search/job_4_results.csv
  ../../../logs/remote/sne_benchmark/sne_benchmark_2817872_5/logs/runs_benchmark/sne_grid_search/job_5_results.csv
  ../../../logs/remote/sne_benchmark/sne_benchmark_2817874_6/logs/runs_benchmark/sne_grid_search/job_6_results.csv
  ../../../logs/remote/sne_benchmark/sne_benchmark_281793

,job_id,source_dir,status,S,N,E,total_envs,wall_time_s,compile_wall_time_s,sim_time_total_s,...,steps_per_sec_per_iter,iter_times_s,ram_per_scene_mean_mb,ram_per_scene_max_mb,vram_allocated_per_scene_mean_mb,vram_allocated_per_scene_max_mb,vram_allocated_total_mb,vram_reserved_per_scene_mean_mb,vram_reserved_per_scene_max_mb,error
0,0,sne_benchmark_2817981_0,ERROR,1,300,32,9600,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,sne_benchmark_2819320_0,OK,4,10,32,1280,442.846685,116.985338,316.820427,...,"[543.3066012093344, 1108.9021949413009, 1083.7...","[58.8986033461988, 28.857369158416986, 29.5277...",2459.550781,2478.062500,4.432495,4.435547,0.0,26.0,26.0,NaN
2,0,sne_benchmark_2819320_0,OK,4,12,128,6144,797.630731,169.938594,614.520804,...,"[1692.3462017578927, 2575.489119982333, 2574.6...","[90.7615710310638, 59.6391570083797, 59.657483...",2630.697266,2637.484375,15.681641,15.681641,0.0,46.0,46.0,NaN
3,0,sne_benchmark_2819320_0,OK,4,16,512,32768,1557.342255,475.518403,1047.615337,...,"[5577.989432614783, 7810.787642684521, 7792.54...","[146.86295302212238, 104.88058790937066, 105.1...",3037.766602,3045.781250,67.326172,67.326172,0.0,136.0,136.0,NaN
4,0,sne_benchmark_2819320_0,OK,4,32,32,4096,1726.054338,288.257500,1425.925556,...,"[453.42307077510765, 742.2250793625436, 724.39...","[225.8376483246684, 137.96354077383876, 141.35...",4460.008789,4481.058594,7.542358,7.545410,0.0,26.0,26.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
453,7,sne_benchmark_2819319_7,OK,1,12,64,768,368.122237,131.384510,228.183543,...,"[389.3379289696116, 938.4925133877332, 961.257...","[49.314486391842365, 20.458341144025326, 19.97...",2635.757812,2635.757812,8.396973,8.396973,0.0,26.0,26.0,NaN
454,7,sne_benchmark_2819319_7,OK,1,4,256,1024,183.613541,71.651299,103.481910,...,"[1204.4394620037622, 2641.8640091476514, 2608....","[21.25470047071576, 9.690127845853567, 9.81294...",1914.125000,1914.125000,25.208984,25.208984,0.0,64.0,64.0,NaN
455,7,sne_benchmark_2819319_7,OK,1,8,1024,8192,401.477708,148.963241,239.310347,...,"[4711.903974476072, 9010.435385623974, 8877.69...","[43.46438321098685, 22.729201335459948, 23.069...",2271.410156,2271.410156,111.414062,111.414062,0.0,256.0,256.0,NaN
456,7,sne_benchmark_2819319_7,OK,1,2,64,128,117.527173,59.273862,48.870366,...,"[262.53221181685075, 673.5570027731902, 684.68...","[12.188980460166931, 4.750897083431482, 4.6736...",1801.800781,1801.800781,6.213867,6.213867,0.0,24.0,24.0,NaN


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Filter for OK status only
ok_data = aggregated[aggregated['status'] == 'OK'].copy()
print(f"Total rows: {len(aggregated)}")
print(f"Rows with OK status: {len(ok_data)}")

# Group by (S, N, E) and average wall_time and total_envs
grouped = ok_data.groupby(['S', 'N', 'E']).agg({
    'wall_time_s': 'mean',
    'total_envs': 'first'  # total_envs is deterministic from S, N, E
}).reset_index()

print(f"Unique (S, N, E) configurations: {len(grouped)}")

# Extract SNE configuration values and metrics
S = grouped['S'].values.astype(float)
N = grouped['N'].values.astype(float)
E = grouped['E'].values.astype(float)
wall_time = grouped['wall_time_s'].values.astype(float)
total_envs = grouped['total_envs'].values.astype(float)

print(f"\nS range: [{S.min():.0f}, {S.max():.0f}]  (parallel scenes)")
print(f"N range: [{N.min():.0f}, {N.max():.0f}]  (URDFs per scene)")
print(f"E range: [{E.min():.0f}, {E.max():.0f}]  (envs per scene)")
print(f"Wall time range: [{wall_time.min():.2f}, {wall_time.max():.2f}] seconds")
print(f"total_envs range: [{total_envs.min():.0f}, {total_envs.max():.0f}]")

fig = plt.figure(figsize=(14, 18))

# ===== Plot 1: Colored by Wall Time =====
ax1 = fig.add_subplot(211, projection='3d')
scatter1 = ax1.scatter(S, N, E, c=wall_time, cmap='viridis', s=80, marker='o',
                       edgecolors='black', linewidth=0.3, alpha=0.8)
cbar1 = fig.colorbar(scatter1, ax=ax1, label='Wall Time (s)', pad=0.12, shrink=0.6)
ax1.set_xlabel('S (parallel scenes)', fontsize=11, fontweight='bold')
ax1.set_ylabel('N (URDFs per scene)', fontsize=11, fontweight='bold')
ax1.set_zlabel('E (envs per scene)', fontsize=11, fontweight='bold')
ax1.set_title('Colored by Wall Time (averaged across runs)', fontsize=13, fontweight='bold')
ax1.view_init(elev=20, azim=45)

# ===== Plot 2: Colored by Total Envs =====
ax2 = fig.add_subplot(212, projection='3d')
scatter2 = ax2.scatter(S, N, E, c=total_envs, cmap='plasma', s=80, marker='o',
                       edgecolors='black', linewidth=0.3, alpha=0.8)
cbar2 = fig.colorbar(scatter2, ax=ax2, label='Total Envs (S × N × E)', pad=0.12, shrink=0.6)
ax2.set_xlabel('S (parallel scenes)', fontsize=11, fontweight='bold')
ax2.set_ylabel('N (URDFs per scene)', fontsize=11, fontweight='bold')
ax2.set_zlabel('E (envs per scene)', fontsize=11, fontweight='bold')
ax2.set_title('Colored by Total Envs', fontsize=13, fontweight='bold')
ax2.view_init(elev=20, azim=45)

fig.suptitle('SNE Configuration Space (OK status only, wall times averaged)', fontsize=14, fontweight='bold', y=0.99)

plt.tight_layout()
plt.show()